# Script for data cleaning
This notebook carry out the first step of data cleaning for combined.csv

## 1. Data Import and Setup
First, let's import the necessary libraries and load our dataset.


In [21]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


# Set plot style
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


## 2. Loading the Data
We'll now load the first chunk of our dataset. Since we identified that the file uses semicolons as delimiters, we'll specify that in our read_csv call.


In [22]:
# Load the first chunk of the dataset
file_path = '/Users/agetman/Desktop/Twork/energyThesis/data/combined_60_chunks.csv'

# First, let's peek at the file to confirm the delimiter
with open(file_path, 'r') as f:
    first_line = f.readline().strip()
    
print(f"First line of the file: {first_line}")

# Load the data with the proper delimiter
df = pd.read_csv(file_path, delimiter=';')

# Display basic information
print(f"\nDataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024**2:.2f} MB")


First line of the file: EAN_ID;Datum;Datum_Startuur;Volume_Afname_kWh;Volume_Injectie_kWh;Warmtepomp_Indicator;Elektrisch_Voertuig_Indicator;PV-Installatie_Indicator;Contract_Categorie

Dataset shape: (6000000, 9)
Memory usage: 411.99 MB


## 3. Column Translation and Data Type Conversion
The data is in Dutch, so let's translate the column names to English for better understanding and ensure proper data type conversion.


In [23]:
# Original column names and their meanings
column_translations = {
    'EAN_ID': 'EAN_ID',  # Unique identifier, keep as is
    'Datum': 'Date',  # Date
    'Datum_Startuur': 'Date_StartHour',  # Date with hour (timestamp)
    'Volume_Afname_kWh': 'Volume_Consumption_kWh',  # Electricity consumption
    'Volume_Injectie_kWh': 'Volume_Injection_kWh',  # Electricity fed back to grid
    'Warmtepomp_Indicator': 'Heat_Pump_Indicator',  # Whether household has heat pump (binary)
    'Elektrisch_Voertuig_Indicator': 'Electric_Vehicle_Indicator',  # Whether household has electric vehicle (binary)
    'PV-Installatie_Indicator': 'PV_Installation_Indicator',  # Whether household has solar panels (binary)
    'Contract_Categorie': 'Contract_Category'  # Category of contract
}

df.rename(columns=column_translations, inplace=True)
df


,EAN_ID,Date,Date_StartHour,Volume_Consumption_kWh,Volume_Injection_kWh,Heat_Pump_Indicator,Electric_Vehicle_Indicator,PV_Installation_Indicator,Contract_Category
0,1,2022-01-01,2022-01-01T00:00:00.000Z,0.760,0.0,0,1,0,Residentieel
1,1,2022-01-01,2022-01-01T00:15:00.000Z,0.789,0.0,0,1,0,Residentieel
2,1,2022-01-01,2022-01-01T00:30:00.000Z,1.131,0.0,0,1,0,Residentieel
3,1,2022-01-01,2022-01-01T00:45:00.000Z,0.791,0.0,0,1,0,Residentieel
4,1,2022-01-01,2022-01-01T01:00:00.000Z,0.791,0.0,0,1,0,Residentieel
...,...,...,...,...,...,...,...,...,...
5999995,172,2022-03-26,2022-03-26T22:45:00.000Z,0.030,0.0,0,0,0,Residentieel
5999996,172,2022-03-26,2022-03-26T23:00:00.000Z,0.027,0.0,0,0,0,Residentieel
5999997,172,2022-03-26,2022-03-26T23:15:00.000Z,0.025,0.0,0,0,0,Residentieel
5999998,172,2022-03-26,2022-03-26T23:30:00.000Z,0.039,0.0,0,0,0,Residentieel


## 4. Exploring the Data Structure
Let's examine the columns and first few rows of our dataset.


In [24]:
# Display column names
print("Columns in the dataset:")
for col in df.columns:
    print(f"- {col}")

Columns in the dataset:
- EAN_ID
- Date
- Date_StartHour
- Volume_Consumption_kWh
- Volume_Injection_kWh
- Heat_Pump_Indicator
- Electric_Vehicle_Indicator
- PV_Installation_Indicator
- Contract_Category


## 5. Convert numeric columns and handle any data type issues.


In [25]:
# For consumption and injection
for col in ['Volume_Consumption_kWh', 'Volume_Injection_kWh']:
    if col in df.columns:
        # Check current type
        current_type = df[col].dtype
        print(f"Converting {col} from {current_type}")
        
        # Convert to numeric
        if not pd.api.types.is_numeric_dtype(df[col]):
            # If string/object, handle comma as decimal separator
            df[col] = pd.to_numeric(df[col].astype(str).map(lambda x: x.replace(',', '.')), errors='coerce')
        else:
            # If already somewhat numeric but needs cleaning
            df[col] = pd.to_numeric(df[col], errors='coerce')


Converting Volume_Consumption_kWh from float64
Converting Volume_Injection_kWh from float64


In [26]:
df['Date_StartHour']=pd.to_datetime(df['Date_StartHour'])
df['Date']=pd.to_datetime(df['Date'])
df['Hour'] = df['Date_StartHour'].dt.hour

# You can also extract other time components
df['Minute'] = df['Date_StartHour'].dt.minute
df['Day'] = df['Date_StartHour'].dt.day
df['Month'] = df['Date_StartHour'].dt.month
df['Year'] = df['Date_StartHour'].dt.year
df['Day_of_Week_Name'] = df['Date_StartHour'].dt.day_name()

In [27]:
for col in ['Heat_Pump_Indicator', 'Electric_Vehicle_Indicator', 'PV_Installation_Indicator']:
    if col in df.columns:
        df[col] = df[col].astype(int)

In [28]:
# Check data types after conversion
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000000 entries, 0 to 5999999
Data columns (total 15 columns):
 #   Column                      Dtype              
---  ------                      -----              
 0   EAN_ID                      int64              
 1   Date                        datetime64[ns]     
 2   Date_StartHour              datetime64[ns, UTC]
 3   Volume_Consumption_kWh      float64            
 4   Volume_Injection_kWh        float64            
 5   Heat_Pump_Indicator         int64              
 6   Electric_Vehicle_Indicator  int64              
 7   PV_Installation_Indicator   int64              
 8   Contract_Category           object             
 9   Hour                        int32              
 10  Minute                      int32              
 11  Day                         int32              
 12  Month                       int32              
 13  Year                        int32              
 14  Day_of_Week_Name            object

In [29]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
if any(missing_values > 0):
    for col in missing_values[missing_values > 0].index:
        print(f"- {col}: {missing_values[col]} missing values")
else:
    print("No missing values found")


Missing values per column:
No missing values found


In [31]:
# Create new variables based on the specified conditions
df['With_HeatPump_Solar'] = ((df['Heat_Pump_Indicator'] == 1) & (df['PV_Installation_Indicator'] == 1)).astype(int)
df['With_Solar_EV'] = ((df['PV_Installation_Indicator'] == 1) & (df['Electric_Vehicle_Indicator'] == 1)).astype(int)
df['Without_Solar_With_EV'] = ((df['PV_Installation_Indicator'] == 0) & (df['Electric_Vehicle_Indicator'] == 1)).astype(int)

In [32]:
df

,EAN_ID,Date,Date_StartHour,Volume_Consumption_kWh,Volume_Injection_kWh,Heat_Pump_Indicator,Electric_Vehicle_Indicator,PV_Installation_Indicator,Contract_Category,Hour,Minute,Day,Month,Year,Day_of_Week_Name,With_HeatPump_Solar,With_Solar_EV,Without_Solar_With_EV
0,1,2022-01-01,2022-01-01 00:00:00+00:00,0.760,0.0,0,1,0,Residentieel,0,0,1,1,2022,Saturday,0,0,1
1,1,2022-01-01,2022-01-01 00:15:00+00:00,0.789,0.0,0,1,0,Residentieel,0,15,1,1,2022,Saturday,0,0,1
2,1,2022-01-01,2022-01-01 00:30:00+00:00,1.131,0.0,0,1,0,Residentieel,0,30,1,1,2022,Saturday,0,0,1
3,1,2022-01-01,2022-01-01 00:45:00+00:00,0.791,0.0,0,1,0,Residentieel,0,45,1,1,2022,Saturday,0,0,1
4,1,2022-01-01,2022-01-01 01:00:00+00:00,0.791,0.0,0,1,0,Residentieel,1,0,1,1,2022,Saturday,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5999995,172,2022-03-26,2022-03-26 22:45:00+00:00,0.030,0.0,0,0,0,Residentieel,22,45,26,3,2022,Saturday,0,0,0
5999996,172,2022-03-26,2022-03-26 23:00:00+00:00,0.027,0.0,0,0,0,Residentieel,23,0,26,3,2022,Saturday,0,0,0
5999997,172,2022-03-26,2022-03-26 23:15:00+00:00,0.025,0.0,0,0,0,Residentieel,23,15,26,3,2022,Saturday,0,0,0
5999998,172,2022-03-26,2022-03-26 23:30:00+00:00,0.039,0.0,0,0,0,Residentieel,23,30,26,3,2022,Saturday,0,0,0


In [33]:

# Create the directory if it doesn't exist
save_dir = os.path.join("/Users/agetman/Desktop/Twork/energyThesis/data/", "data_clean")
os.makedirs(save_dir, exist_ok=True)

# Save the DataFrame as pickle
save_path = os.path.join(save_dir, "clean_v2.pkl")
df.to_pickle(save_path)

print(f"DataFrame saved to {save_path} with all data types preserved")


DataFrame saved to /Users/agetman/Desktop/Twork/energyThesis/data/data_clean/clean_v2.pkl with all data types preserved
